# Series and DataFrame

we learn the two core objects of pandas. Everything else in pandas (filtering, groupby, merge, cleaning) is built on these two.

**Contents**
1. Series (1D)
2. DataFrame (2D)
3. Series vs DataFrame vs NumPy
4. Mini AI/ML example (dataset -> X, y)
5. Pandas 3.0 things to remember
6. Common mistakes
7. Interview questions
8. Practice questions with solutions
9. Cheat sheet

In [2]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.6
numpy : 2.5.3


---
# 1. Series

## 1.1 What is a Series?

A **Series** is a **1D labeled array**. It has two parts:

```
index (labels)    values (NumPy array inside)
   "Ravi"    ->        85
   "Asha"    ->        92
   "Karan"   ->        78
```

- `values` -> the data (stored as a NumPy array, so it is fast)
- `index` -> the label of each value (this is what NumPy does not have)
- `dtype` -> one data type for the whole Series
- `name` -> optional name (when a Series becomes a DataFrame column, this is the column name)

**Think of it as:** one column of an Excel sheet, or a NumPy array that has names for its positions.

In AI/ML, one Series is usually **one feature** (like `age`) or **the target** (like `price`).

## 1.2 Creating a Series

In [3]:
# From a list (default index = 0, 1, 2, ...)
s1 = pd.Series([10, 20, 30, 40])
print(s1)

0    10
1    20
2    30
3    40
dtype: int64


In [4]:
# From a list with a custom index and a name
marks = pd.Series([85, 92, 78], index=["Ravi", "Asha", "Karan"], name="maths")
print(marks)

Ravi     85
Asha     92
Karan    78
Name: maths, dtype: int64


In [5]:
# From a dict (keys become the index)
d = {"Ravi": 85, "Asha": 92, "Karan": 78}
print(pd.Series(d))

Ravi     85
Asha     92
Karan    78
dtype: int64


In [6]:
# From a NumPy array
arr = np.array([1.5, 2.5, 3.5])
print(pd.Series(arr, index=["a", "b", "c"]))

a    1.5
b    2.5
c    3.5
dtype: float64


In [7]:
# From a single value (scalar) -> it is repeated for every index (broadcasting)
print(pd.Series(0, index=["a", "b", "c"]))

a    0
b    0
c    0
dtype: int64


## 1.3 Important attributes

In [8]:
print("values :", marks.values)      # NumPy array inside
print("index  :", marks.index)       # labels
print("dtype  :", marks.dtype)
print("shape  :", marks.shape)       # (3,)
print("size   :", marks.size)        # number of elements
print("ndim   :", marks.ndim)        # always 1 for a Series
print("name   :", marks.name)
print("type of values:", type(marks.values))

values : [85 92 78]
index  : Index(['Ravi', 'Asha', 'Karan'], dtype='str')
dtype  : int64
shape  : (3,)
size   : 3
ndim   : 1
name   : maths
type of values: <class 'numpy.ndarray'>


## 1.4 Indexing and slicing

There are **two ways** to access data:

| Style | Meaning | End of slice |
|---|---|---|
| `.loc[...]` | by **label** | **included** |
| `.iloc[...]` | by **position** (0, 1, 2...) | **excluded** (like normal Python) |

Always prefer `.loc` and `.iloc`. They are clear and safe.

In [9]:
print(marks.loc["Asha"])       # by label
print(marks.iloc[1])           # by position

92
92


In [10]:
# Slicing
print(marks.loc["Ravi":"Asha"])   # label slice -> Asha IS included
print()
print(marks.iloc[0:2])            # position slice -> index 2 NOT included

Ravi    85
Asha    92
Name: maths, dtype: int64

Ravi    85
Asha    92
Name: maths, dtype: int64


In [11]:
# Multiple items (pass a list)
print(marks.loc[["Karan", "Ravi"]])

Karan    78
Ravi     85
Name: maths, dtype: int64


In [12]:
# Boolean filtering (VERY important, used everywhere in ML data cleaning)
print(marks > 80)
print()
print(marks[marks > 80])

Ravi      True
Asha      True
Karan    False
Name: maths, dtype: bool

Ravi    85
Asha    92
Name: maths, dtype: int64


In [13]:
# Multiple conditions: use & (and), | (or), ~ (not) and put each condition in ( )
print(marks[(marks > 80) & (marks < 90)])

Ravi    85
Name: maths, dtype: int64


> **Warning:** `marks[0]` (plain `[]` with an integer on a string index) is confusing. In older pandas it silently used position, in newer versions it raises an error. So always write `marks.iloc[0]` or `marks.loc["Ravi"]`.

## 1.5 Vectorized operations

Just like NumPy, operations work on the **whole Series at once**. No loop needed.

In [14]:
print(marks + 5)         # add to every element
print(marks * 2)
print(marks / 100)

Ravi     90
Asha     97
Karan    83
Name: maths, dtype: int64
Ravi     170
Asha     184
Karan    156
Name: maths, dtype: int64
Ravi     0.85
Asha     0.92
Karan    0.78
Name: maths, dtype: float64


In [15]:
# NumPy functions work directly on a Series and keep the index
print(np.sqrt(marks))
print()
print(marks.mean(), marks.max(), marks.min(), marks.std())

Ravi     9.219544
Asha     9.591663
Karan    8.831761
Name: maths, dtype: float64

85.0 92 78 7.0


## 1.6 Index alignment (the BIG difference from NumPy)

When you add two Series, pandas matches them **by index label**, not by position.
If a label is missing on one side, the result is `NaN`.

This is a very common source of surprise bugs, so understand it well.

In [16]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])

print(a + b)

w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64


In [17]:
# x and w only exist on one side -> NaN.
# Fix: use the method version and give a fill_value
print(a.add(b, fill_value=0))

w    30.0
x     1.0
y    12.0
z    23.0
dtype: float64


## 1.7 Missing values (NaN)

Real datasets always have missing values. Pandas has good tools for them.

In [18]:
s = pd.Series([10, np.nan, 30, None, 50])
print(s)
print("dtype:", s.dtype)   # float64, because NaN is a float

0    10.0
1     NaN
2    30.0
3     NaN
4    50.0
dtype: float64
dtype: float64


In [19]:
print(s.isna())            # True where missing
print("missing count:", s.isna().sum())
print()
print(s.dropna())          # remove missing
print()
print(s.fillna(0))         # fill with a value
print()
print(s.fillna(s.mean()))  # fill with mean (very common in ML)

0    False
1     True
2    False
3     True
4    False
dtype: bool
missing count: 2

0    10.0
2    30.0
4    50.0
dtype: float64

0    10.0
1     0.0
2    30.0
3     0.0
4    50.0
dtype: float64

0    10.0
1    30.0
2    30.0
3    30.0
4    50.0
dtype: float64


## 1.8 Useful Series methods

In [20]:
s = pd.Series([3, 1, 2, 3, 3, 2, 5], name="rating")

print(s.head(3))              # first 3
print(s.tail(2))              # last 2
print(s.unique())             # unique values
print(s.nunique())            # count of unique values
print(s.value_counts())       # frequency of each value (used a lot in EDA)

0    3
1    1
2    2
Name: rating, dtype: int64
5    2
6    5
Name: rating, dtype: int64
[3 1 2 5]
4
rating
3    3
2    2
1    1
5    1
Name: count, dtype: int64


In [21]:
print(s.sort_values())                  # sort by value
print(s.sort_values(ascending=False))
print(s.sort_index())                   # sort by index
print("idxmax:", s.idxmax(), "| idxmin:", s.idxmin())   # label of max / min

1    1
2    2
5    2
0    3
3    3
4    3
6    5
Name: rating, dtype: int64
6    5
0    3
3    3
4    3
2    2
5    2
1    1
Name: rating, dtype: int64
0    3
1    1
2    2
3    3
4    3
5    2
6    5
Name: rating, dtype: int64
idxmax: 6 | idxmin: 1


In [22]:
print(s.describe())     # count, mean, std, min, quartiles, max

count    7.000000
mean     2.714286
std      1.253566
min      1.000000
25%      2.000000
50%      3.000000
75%      3.000000
max      5.000000
Name: rating, dtype: float64


In [23]:
print(s.isin([1, 5]))          # membership check
print(s.between(2, 3))         # inclusive range check

0    False
1     True
2    False
3    False
4    False
5    False
6     True
Name: rating, dtype: bool
0     True
1    False
2     True
3     True
4     True
5     True
6    False
Name: rating, dtype: bool


In [24]:
# map(): replace values using a dict or function (element-wise)
grade = pd.Series(["A", "B", "A", "C"])
print(grade.map({"A": 4, "B": 3, "C": 2}))

0    4
1    3
2    4
3    2
dtype: int64


In [25]:
# apply(): run a function on each element
marks = pd.Series([85, 92, 78], index=["Ravi", "Asha", "Karan"], name="maths")
print(marks.apply(lambda x: "pass" if x >= 80 else "fail"))

Ravi     pass
Asha     pass
Karan    fail
Name: maths, dtype: str


In [26]:
# astype(): change dtype
print(marks.astype("float64"))

Ravi     85.0
Asha     92.0
Karan    78.0
Name: maths, dtype: float64


## 1.9 Series vs NumPy array

| | NumPy ndarray | pandas Series |
|---|---|---|
| Labels | No (positions only) | Yes (index) |
| Missing values | Weak | Built-in |
| Alignment on operations | By position | By label |
| Mixed types | No | Possible (object/str dtype) |
| Speed for pure math | Fastest | Slight overhead |
| Convert | `s.to_numpy()` | `pd.Series(arr)` |

---
# 2. DataFrame

## 2.1 What is a DataFrame?

A **DataFrame** is a **2D labeled table**: rows and columns.

```
          columns ->
          name    age   city     score
index  0  Ravi    21    Indore   85
  |    1  Asha    22    Bhopal   92
  v    2  Karan   23    Delhi    78
```

- Every **column is a Series** (and all columns share the same index).
- Each column can have its **own dtype** (this is the big win over NumPy).
- It has `index` (row labels) and `columns` (column labels).

In AI/ML: the **DataFrame is your dataset**. Columns = features + target, rows = samples.

## 2.2 Creating a DataFrame

In [75]:
# Method 1: dict of lists (most common)
df = pd.DataFrame({
    "name":  ["Ravi", "Asha", "Karan", "Neha", "Amit"],
    "age":   [21, 22, 23, 21, 24],
    "city":  ["Indore", "Bhopal", "Delhi", "Indore", "Pune"],
    "hours_studied": [5.5, 7.0, 3.0, 6.5, 4.0],
    "score": [85, 92, 58, 88, 64],
})
df

,name,age,city,hours_studied,score
0,Ravi,21,Indore,5.5,85
1,Asha,22,Bhopal,7.0,92
2,Karan,23,Delhi,3.0,58
3,Neha,21,Indore,6.5,88
4,Amit,24,Pune,4.0,64


In [28]:
# Method 2: list of dicts (like JSON records from an API)
records = [
    {"name": "Ravi", "age": 21},
    {"name": "Asha", "age": 22},
    {"name": "Karan"},            # age missing -> NaN
]
pd.DataFrame(records)

,name,age
0,Ravi,21.0
1,Asha,22.0
2,Karan,NaN


In [29]:
# Method 3: dict of Series (aligned by index)
x = pd.Series([1, 2, 3], index=["a", "b", "c"])
y = pd.Series([10, 20, 30], index=["a", "b", "c"])
pd.DataFrame({"x": x, "y": y})

,x,y
a,1,10
b,2,20
c,3,30


In [30]:
# Method 4: from a 2D NumPy array (you must give column names)
arr = np.array([[1, 2, 3],
                [4, 5, 6]])
pd.DataFrame(arr, columns=["f1", "f2", "f3"], index=["row1", "row2"])

,f1,f2,f3
row1,1,2,3
row2,4,5,6


In [31]:
# Method 5: from a file (we study this in detail in the next chapter)
# df = pd.read_csv("data.csv")

## 2.3 Important attributes

In [32]:
print("shape   :", df.shape)       # (rows, columns)
print("size    :", df.size)        # rows * columns
print("ndim    :", df.ndim)        # 2
print("columns :", df.columns.tolist())
print("index   :", df.index)
print()
print(df.dtypes)                   # dtype of EACH column

shape   : (5, 5)
size    : 25
ndim    : 2
columns : ['name', 'age', 'city', 'hours_studied', 'score']
index   : RangeIndex(start=0, stop=5, step=1)

name                 str
age                int64
city                 str
hours_studied    float64
score              int64
dtype: object


## 2.4 First look at any dataset (do this every time)

Whenever you get a new dataset, run these 5 things first:
`head()`, `shape`, `info()`, `describe()`, `isna().sum()`

In [33]:
df.head(3)      # first 3 rows (default 5)

,name,age,city,hours_studied,score
0,Ravi,21,Indore,5.5,85
1,Asha,22,Bhopal,7.0,92
2,Karan,23,Delhi,3.0,58


In [34]:
df.tail(2)      # last 2 rows

,name,age,city,hours_studied,score
3,Neha,21,Indore,6.5,88
4,Amit,24,Pune,4.0,64


In [35]:
df.sample(2, random_state=42)   # 2 random rows (random_state makes it repeatable)

,name,age,city,hours_studied,score
1,Asha,22,Bhopal,7.0,92
4,Amit,24,Pune,4.0,64


In [36]:
df.info()       # columns, non-null counts, dtypes, memory

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           5 non-null      str    
 1   age            5 non-null      int64  
 2   city           5 non-null      str    
 3   hours_studied  5 non-null      float64
 4   score          5 non-null      int64  
dtypes: float64(1), int64(2), str(2)
memory usage: 332.0 bytes


In [37]:
df.describe()   # stats of numeric columns only (by default)

,age,hours_studied,score
count,5.00000,5.000000,5.000000
mean,22.20000,5.200000,77.400000
std,1.30384,1.680774,15.323185
min,21.00000,3.000000,58.000000
25%,21.00000,4.000000,64.000000
50%,22.00000,5.500000,85.000000
75%,23.00000,6.500000,88.000000
max,24.00000,7.000000,92.000000


In [38]:
df.describe(include="all")   # include text columns too

,name,age,city,hours_studied,score
count,5,5.00000,5,5.000000,5.000000
unique,5,NaN,4,NaN,NaN
top,Ravi,NaN,Indore,NaN,NaN
freq,1,NaN,2,NaN,NaN
mean,NaN,22.20000,NaN,5.200000,77.400000
std,NaN,1.30384,NaN,1.680774,15.323185
min,NaN,21.00000,NaN,3.000000,58.000000
25%,NaN,21.00000,NaN,4.000000,64.000000
50%,NaN,22.00000,NaN,5.500000,85.000000
75%,NaN,23.00000,NaN,6.500000,88.000000


In [39]:
df.isna().sum()   # missing values per column

name             0
age              0
city             0
hours_studied    0
score            0
dtype: int64

## 2.5 Selecting columns

- `df["col"]` -> returns a **Series**
- `df[["col1", "col2"]]` -> returns a **DataFrame** (note the double brackets)

In [40]:
col = df["score"]
print(type(col))
print(col)

<class 'pandas.Series'>
0    85
1    92
2    58
3    88
4    64
Name: score, dtype: int64


In [41]:
sub = df[["name", "score"]]
print(type(sub))
sub

<class 'pandas.DataFrame'>


,name,score
0,Ravi,85
1,Asha,92
2,Karan,58
3,Neha,88
4,Amit,64


## 2.6 Selecting rows (quick look, full detail comes in the next chapters)

Same rule as Series: `.loc` = labels, `.iloc` = positions.
Syntax: `df.loc[row_selection, column_selection]`

In [42]:
df.loc[0]                         # row with label 0 (returns a Series)

name               Ravi
age                  21
city             Indore
hours_studied       5.5
score                85
Name: 0, dtype: object

In [43]:
df.loc[1:3, ["name", "score"]]    # rows 1 to 3 (3 included), two columns

,name,score
1,Asha,92
2,Karan,58
3,Neha,88


In [44]:
df.iloc[0:2, 0:3]                 # first 2 rows, first 3 columns (end excluded)

,name,age,city
0,Ravi,21,Indore
1,Asha,22,Bhopal


In [45]:
df[df["score"] > 80]              # boolean filtering

,name,age,city,hours_studied,score
0,Ravi,21,Indore,5.5,85
1,Asha,22,Bhopal,7.0,92
3,Neha,21,Indore,6.5,88


In [46]:
df[(df["city"] == "Indore") & (df["score"] > 85)]

,name,age,city,hours_studied,score
3,Neha,21,Indore,6.5,88


## 2.7 Adding, changing, renaming and dropping

In [47]:
# Add a new column (vectorized, no loop)
df["passed"] = df["score"] >= 60
df["score_pct"] = df["score"] / 100
df

,name,age,city,hours_studied,score,passed,score_pct
0,Ravi,21,Indore,5.5,85,True,0.85
1,Asha,22,Bhopal,7.0,92,True,0.92
2,Karan,23,Delhi,3.0,58,False,0.58
3,Neha,21,Indore,6.5,88,True,0.88
4,Amit,24,Pune,4.0,64,True,0.64


In [48]:
# Change a column
df["age"] = df["age"] + 1
df[["name", "age"]]

,name,age
0,Ravi,22
1,Asha,23
2,Karan,24
3,Neha,22
4,Amit,25


In [49]:
# Rename columns (returns a new DataFrame, so we assign it back)
df = df.rename(columns={"hours_studied": "study_hours"})
df.columns.tolist()

['name', 'age', 'city', 'study_hours', 'score', 'passed', 'score_pct']

In [50]:
# Drop a column (returns a new DataFrame)
df = df.drop(columns=["score_pct"])
df.columns.tolist()

['name', 'age', 'city', 'study_hours', 'score', 'passed']

In [51]:
# Drop a row by label
df.drop(index=4)

,name,age,city,study_hours,score,passed
0,Ravi,22,Indore,5.5,85,True
1,Asha,23,Bhopal,7.0,92,True
2,Karan,24,Delhi,3.0,58,False
3,Neha,22,Indore,6.5,88,True


> `drop`, `rename`, `fillna` etc. return a **new** object. The original does not change unless you assign it back (`df = df.drop(...)`). Avoid `inplace=True`, it is not recommended anymore.

## 2.8 Index operations

In [52]:
# Make a column the index
d2 = df.set_index("name")
d2

,age,city,study_hours,score,passed
name,,,,,
Ravi,22,Indore,5.5,85,True
Asha,23,Bhopal,7.0,92,True
Karan,24,Delhi,3.0,58,False
Neha,22,Indore,6.5,88,True
Amit,25,Pune,4.0,64,True


In [53]:
d2.loc["Asha"]        # now we can access by name

age                23
city           Bhopal
study_hours       7.0
score              92
passed           True
Name: Asha, dtype: object

In [54]:
# Move the index back to a normal column
d2.reset_index()

,name,age,city,study_hours,score,passed
0,Ravi,22,Indore,5.5,85,True
1,Asha,23,Bhopal,7.0,92,True
2,Karan,24,Delhi,3.0,58,False
3,Neha,22,Indore,6.5,88,True
4,Amit,25,Pune,4.0,64,True


## 2.9 Sorting

In [55]:
df.sort_values("score", ascending=False)

,name,age,city,study_hours,score,passed
1,Asha,23,Bhopal,7.0,92,True
3,Neha,22,Indore,6.5,88,True
0,Ravi,22,Indore,5.5,85,True
4,Amit,25,Pune,4.0,64,True
2,Karan,24,Delhi,3.0,58,False


In [56]:
df.sort_values(["city", "score"], ascending=[True, False])

,name,age,city,study_hours,score,passed
1,Asha,23,Bhopal,7.0,92,True
2,Karan,24,Delhi,3.0,58,False
3,Neha,22,Indore,6.5,88,True
0,Ravi,22,Indore,5.5,85,True
4,Amit,25,Pune,4.0,64,True


## 2.10 Missing values in a DataFrame

In [57]:
dfm = pd.DataFrame({
    "age":    [21, np.nan, 23, 24],
    "salary": [30000, 40000, np.nan, np.nan],
    "city":   ["Indore", "Delhi", None, "Pune"],
})
print(dfm)
print()
print(dfm.isna().sum())

    age   salary    city
0  21.0  30000.0  Indore
1   NaN  40000.0   Delhi
2  23.0      NaN     NaN
3  24.0      NaN    Pune

age       1
salary    2
city      1
dtype: int64


In [58]:
print(dfm.dropna())                         # drop rows having ANY missing value
print()
print(dfm.fillna({"age": dfm["age"].mean(), "salary": 0, "city": "Unknown"}))

    age   salary    city
0  21.0  30000.0  Indore

         age   salary     city
0  21.000000  30000.0   Indore
1  22.666667  40000.0    Delhi
2  23.000000      0.0  Unknown
3  24.000000      0.0     Pune


## 2.11 Converting between pandas, NumPy and Python

In [59]:
nums = df[["age", "study_hours", "score"]]

arr = nums.to_numpy()          # DataFrame -> NumPy 2D array
print(arr)
print(arr.shape)

[[22.   5.5 85. ]
 [23.   7.  92. ]
 [24.   3.  58. ]
 [22.   6.5 88. ]
 [25.   4.  64. ]]
(5, 3)


In [60]:
print(df["score"].tolist())              # Series -> Python list
print(df.head(2).to_dict(orient="records"))   # DataFrame -> list of dicts (JSON like)

[85, 92, 58, 88, 64]
[{'name': 'Ravi', 'age': 22, 'city': 'Indore', 'study_hours': 5.5, 'score': 85, 'passed': True}, {'name': 'Asha', 'age': 23, 'city': 'Bhopal', 'study_hours': 7.0, 'score': 92, 'passed': True}]


---
# 3. Series vs DataFrame vs NumPy

| Feature | NumPy ndarray | Series | DataFrame |
|---|---|---|---|
| Dimensions | N-D | 1D | 2D |
| Labels | No | Row labels (index) | Row + column labels |
| Types | One dtype | One dtype | One dtype **per column** |
| Missing data | Weak | Good | Good |
| One column of it | - | - | is a Series |
| Used in ML as | tensors, final model input | one feature / target | the whole dataset |

**Relationship:** `DataFrame` = many `Series` sharing one index. Each `Series` = a NumPy array + labels.

---
# 4. Mini AI/ML example: from a table to model input

In ML we always do this: load data in a DataFrame -> inspect -> split into **features (X)** and **target (y)** -> give to the model as NumPy arrays.

Problem: predict house price from area and rooms.

In [61]:
houses = pd.DataFrame({
    "area_sqft": [850, 1200, 1500, 700, 2000, 1100],
    "bedrooms":  [2, 3, 3, 1, 4, 2],
    "age_years": [10, 5, np.nan, 15, 2, 8],     # one value missing
    "price_lakh": [45, 70, 85, 30, 120, 60],
})
houses

,area_sqft,bedrooms,age_years,price_lakh
0,850,2,10.0,45
1,1200,3,5.0,70
2,1500,3,NaN,85
3,700,1,15.0,30
4,2000,4,2.0,120
5,1100,2,8.0,60


In [62]:
# 1. Inspect
houses.info()
print(houses.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   area_sqft   6 non-null      int64  
 1   bedrooms    6 non-null      int64  
 2   age_years   5 non-null      float64
 3   price_lakh  6 non-null      int64  
dtypes: float64(1), int64(3)
memory usage: 324.0 bytes
area_sqft     0
bedrooms      0
age_years     1
price_lakh    0
dtype: int64


In [63]:
# 2. Clean: fill the missing age with the median
houses["age_years"] = houses["age_years"].fillna(houses["age_years"].median())
houses

,area_sqft,bedrooms,age_years,price_lakh
0,850,2,10.0,45
1,1200,3,5.0,70
2,1500,3,8.0,85
3,700,1,15.0,30
4,2000,4,2.0,120
5,1100,2,8.0,60


In [64]:
# 3. Quick feel of the data: which feature relates most to price?
houses.corr()["price_lakh"].sort_values(ascending=False)

price_lakh    1.000000
area_sqft     0.997267
bedrooms      0.961636
age_years    -0.897015
Name: price_lakh, dtype: float64

In [65]:
# 4. Split into features (X) and target (y)
X = houses[["area_sqft", "bedrooms", "age_years"]]    # DataFrame (2D)
y = houses["price_lakh"]                              # Series (1D)

print("X shape:", X.shape, "| y shape:", y.shape)

# scikit-learn accepts DataFrame/Series directly, but this is the NumPy form:
X_np = X.to_numpy()
y_np = y.to_numpy()
print(type(X_np), X_np.shape)

X shape: (6, 3) | y shape: (6,)
<class 'numpy.ndarray'> (6, 3)


That is the whole story of how pandas and NumPy work together in ML:

**pandas** = load, clean, explore, prepare -> **NumPy** = fast numbers for the model.

---
# 5. Pandas 3.0 things to remember

1. **Copy-on-Write is the default.** Changing a subset (slice) never changes the original DataFrame behind your back. Also, "chained assignment" like `df[df["a"] > 1]["b"] = 0` does **not** work. Use `df.loc[df["a"] > 1, "b"] = 0`.
2. **Text columns get a dedicated string dtype** (shown as `str`) instead of the old `object`. Mixed or non-text data still uses `object`.
3. **Use `.loc` / `.iloc`** for access. Do not rely on `s[0]` positional fallback on label indexes.
4. Avoid `inplace=True`. Assign the result back.

In [66]:
# Safe way to update rows by condition
tmp = df.copy()
tmp.loc[tmp["score"] < 60, "passed"] = False
tmp[["name", "score", "passed"]]

,name,score,passed
0,Ravi,85,True
1,Asha,92,True
2,Karan,58,False
3,Neha,88,True
4,Amit,64,True


---
# 6. Common mistakes

| Mistake | Correct way |
|---|---|
| `df["a", "b"]` | `df[["a", "b"]]` (list inside brackets) |
| `df[a > 1 and b < 5]` | `df[(a > 1) & (b < 5)]` (use `&`, `|`, `~` with brackets) |
| `df.drop("col")` and expecting change | `df = df.drop(columns="col")` |
| Looping over rows with `for` | Use vectorized operations |
| Forgetting `.loc` end is inclusive | `loc` includes the end, `iloc` excludes it |
| `a + b` on different indexes, surprised by NaN | Check indexes, or use `a.add(b, fill_value=0)` |
| Chained assignment `df[cond]["x"] = 1` | `df.loc[cond, "x"] = 1` |

---
# 7. Interview questions (with short answers)

**Q1. What is the difference between a Series and a DataFrame?**
Series is 1D labeled data with one dtype. DataFrame is a 2D table where each column is a Series and can have its own dtype.

**Q2. Why use pandas when we have NumPy?**
NumPy has one dtype and position-only access. Pandas adds labels, mixed types, missing value handling, file I/O, groupby, merge and time series.

**Q3. Difference between `loc` and `iloc`?**
`loc` uses labels and includes the end of a slice. `iloc` uses integer positions and excludes the end.

**Q4. What happens when you add two Series with different indexes?**
Pandas aligns on index labels. Labels present in only one Series give `NaN`. Use `add(..., fill_value=0)` to avoid it.

**Q5. `df["a"]` vs `df[["a"]]`?**
First returns a Series, second returns a DataFrame with one column.

**Q6. Why is pandas fast?**
Columns are NumPy arrays under the hood, and operations are vectorized in C.

**Q7. How do you handle missing values?**
Detect with `isna().sum()`, then `dropna()` or `fillna()` (mean, median, constant, forward fill), depending on the problem.

**Q8. Is a DataFrame mutable? Does `df.drop()` change it?**
Yes it is mutable, but `drop` returns a new DataFrame. You have to assign it back.

**Q9. How do you convert a DataFrame into a NumPy array for a model?**
`df.to_numpy()` (preferred over `.values`).

**Q10. What is `describe()` used for?**
Quick statistics (count, mean, std, min, quartiles, max) for numeric columns. First step of EDA.

---
# 8. Practice questions

Try each one yourself first, then check the solution below.

Use this dataset:

In [67]:
emp = pd.DataFrame({
    "name":   ["A", "B", "C", "D", "E", "F"],
    "dept":   ["ML", "Web", "ML", "Data", "Web", "ML"],
    "exp":    [2, 5, 3, 1, 8, 4],
    "salary": [12, 18, 15, 9, 25, 16],       # in LPA
})
emp

,name,dept,exp,salary
0,A,ML,2,12
1,B,Web,5,18
2,C,ML,3,15
3,D,Data,1,9
4,E,Web,8,25
5,F,ML,4,16


**Q1.** Create a Series of salaries with employee names as the index.
**Q2.** Print the name and salary of employees with salary above 14.
**Q3.** Add a column `salary_per_exp` = salary / exp.
**Q4.** Show the top 3 highest paid employees.
**Q5.** Count how many employees are in each department.
**Q6.** Rename `exp` to `experience`, then drop the `salary_per_exp` column.
**Q7.** Convert `exp` and `salary` columns into a NumPy array and print its shape.

### Solutions

In [68]:
# Q1
sal = pd.Series(emp["salary"].to_numpy(), index=emp["name"], name="salary")
print(sal)

name
A    12
B    18
C    15
D     9
E    25
F    16
Name: salary, dtype: int64


In [69]:
# Q2
emp.loc[emp["salary"] > 14, ["name", "salary"]]

,name,salary
1,B,18
2,C,15
4,E,25
5,F,16


In [70]:
# Q3
emp["salary_per_exp"] = emp["salary"] / emp["exp"]
emp

,name,dept,exp,salary,salary_per_exp
0,A,ML,2,12,6.000
1,B,Web,5,18,3.600
2,C,ML,3,15,5.000
3,D,Data,1,9,9.000
4,E,Web,8,25,3.125
5,F,ML,4,16,4.000


In [71]:
# Q4
emp.sort_values("salary", ascending=False).head(3)
# or: emp.nlargest(3, "salary")

,name,dept,exp,salary,salary_per_exp
4,E,Web,8,25,3.125
1,B,Web,5,18,3.600
5,F,ML,4,16,4.000


In [72]:
# Q5
emp["dept"].value_counts()

dept
ML      3
Web     2
Data    1
Name: count, dtype: int64

In [73]:
# Q6
emp = emp.rename(columns={"exp": "experience"}).drop(columns="salary_per_exp")
emp

,name,dept,experience,salary
0,A,ML,2,12
1,B,Web,5,18
2,C,ML,3,15
3,D,Data,1,9
4,E,Web,8,25
5,F,ML,4,16


In [74]:
# Q7
arr = emp[["experience", "salary"]].to_numpy()
print(arr.shape)

(6, 2)


---
# 9. Cheat sheet

```python
# Create
pd.Series(data, index=..., name=...)
pd.DataFrame(dict_or_list_or_array, columns=..., index=...)

# Inspect
df.head()  df.tail()  df.sample()  df.shape  df.dtypes
df.info()  df.describe()  df.isna().sum()  df.columns  df.index

# Select
df["a"]            # Series
df[["a", "b"]]     # DataFrame
df.loc[rows, cols] # labels (end included)
df.iloc[rows, cols]# positions (end excluded)
df[df["a"] > 5]    # boolean filter

# Modify
df["new"] = ...
df = df.rename(columns={"old": "new"})
df = df.drop(columns=["a"])
df = df.set_index("a")   df = df.reset_index()
df = df.sort_values("a", ascending=False)

# Missing
df.isna()  df.dropna()  df.fillna(value)

# Convert
df.to_numpy()   s.tolist()   df.to_dict(orient="records")
```

**Next chapter:** reading and writing data (CSV, Excel, JSON) and then inspecting real datasets.